# B3b Defense – 04 XGBoost Training

**Objective:** Train XGBoost on `defense_feature_matrix.csv` using `TimeSeriesSplit(n_splits=5)`.
Report MAE, RMSE, sMAPE, and WMAPE per fold. Save the best model to disk.

In [1]:
import os
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib

In [2]:
DATA_PROCESSED = '../data/processed/'
MODELS_DIR     = '../models/'

os.makedirs(MODELS_DIR, exist_ok=True)

In [3]:
# Load feature matrix — explicit to_datetime() to avoid slow dateutil fallback in pandas 2.x
df = pd.read_csv(DATA_PROCESSED + 'defense_feature_matrix.csv', index_col='date')
df.index = pd.to_datetime(df.index)

# Define feature columns and target.
# ADEFNO is explicitly retained as a feature (leading indicator with lags 1–24).
# Only FDEFX itself is excluded from features to avoid target leakage.
feature_cols = [col for col in df.columns if col != 'FDEFX']
target_col   = 'FDEFX'

X = df[feature_cols]
y = df[target_col]

print(f'Feature matrix shape: {X.shape}')
print(f'Target shape:         {y.shape}')
print(f'Target col:           {target_col}')
print(f'FDEFX in feature_cols: {"FDEFX" in feature_cols}  (must be False)')
print(f'ADEFNO in feature_cols: {"ADEFNO" in feature_cols}  (must be True)')

Feature matrix shape: (287, 87)
Target shape:         (287,)
Target col:           FDEFX
FDEFX in feature_cols: False  (must be False)
ADEFNO in feature_cols: True  (must be True)


In [4]:
def smape(actual, pred):
    """Symmetric Mean Absolute Percentage Error (in %)"""
    numerator   = 2 * np.abs(actual - pred)
    denominator = np.abs(actual) + np.abs(pred)
    # Avoid division by zero
    mask = denominator != 0
    return np.mean(numerator[mask] / denominator[mask]) * 100


def wmape(actual, pred):
    """Weighted Mean Absolute Percentage Error (in %)"""
    return np.sum(np.abs(actual - pred)) / np.sum(np.abs(actual)) * 100

In [5]:
tscv = TimeSeriesSplit(n_splits=5)

fold_results = []
best_model   = None

for fold_idx, (train_idx, test_idx) in enumerate(tscv.split(X), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = xgb.XGBRegressor(
        n_estimators  = 200,
        max_depth      = 4,
        learning_rate  = 0.05,
        subsample      = 0.8,
        random_state   = 42,
        verbosity      = 0
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mae   = mean_absolute_error(y_test, y_pred)
    rmse  = np.sqrt(mean_squared_error(y_test, y_pred))
    smape_val = smape(y_test.values, y_pred)
    wmape_val = wmape(y_test.values, y_pred)

    fold_results.append({
        'Fold':  fold_idx,
        'MAE':   round(mae, 4),
        'RMSE':  round(rmse, 4),
        'sMAPE': round(smape_val, 2),
        'WMAPE': round(wmape_val, 2)
    })

    print(f'Fold {fold_idx}: MAE={mae:.4f}  RMSE={rmse:.4f}  sMAPE={smape_val:.2f}%  WMAPE={wmape_val:.2f}%')

    # Keep the last fold's model as the final model
    best_model = model

print('\nCross-validation complete.')

Fold 1: MAE=102.5127  RMSE=119.2993  sMAPE=14.69%  WMAPE=14.01%
Fold 2: MAE=14.4834  RMSE=17.2390  sMAPE=1.82%  WMAPE=1.80%
Fold 3: MAE=23.2702  RMSE=26.2682  sMAPE=3.20%  WMAPE=3.15%
Fold 4: MAE=40.8539  RMSE=50.5019  sMAPE=4.74%  WMAPE=4.73%
Fold 5: MAE=134.4355  RMSE=157.2947  sMAPE=13.45%  WMAPE=12.89%

Cross-validation complete.


In [6]:
results_df = pd.DataFrame(fold_results).set_index('Fold')
print(results_df)
print('\n--- Mean across folds ---')
print(results_df.mean().round(4))

           MAE      RMSE  sMAPE  WMAPE
Fold                                  
1     102.5127  119.2993  14.69  14.01
2      14.4834   17.2390   1.82   1.80
3      23.2702   26.2682   3.20   3.15
4      40.8539   50.5019   4.74   4.73
5     134.4355  157.2947  13.45  12.89

--- Mean across folds ---
MAE      63.1111
RMSE     74.1206
sMAPE     7.5800
WMAPE     7.3160
dtype: float64


In [7]:
model_path = MODELS_DIR + 'xgboost_defense_market.pkl'
joblib.dump(best_model, model_path)
print(f'Model saved to: {model_path}')

Model saved to: ../models/xgboost_defense_market.pkl


## Performance Assessment

**Data basis:** 287 observations, 87 features, target FDEFX in USD billions (SAAR).

| Fold | MAE (USD bn) | RMSE (USD bn) | sMAPE | WMAPE |
|------|-------------|--------------|-------|-------|
| 1 | 103 | 119 | 14.69 % | 14.01 % |
| 2 | 14 | 17 | 1.82 % | 1.80 % |
| 3 | 23 | 26 | 3.20 % | 3.15 % |
| 4 | 41 | 51 | 4.74 % | 4.73 % |
| 5 | 134 | 157 | 13.45 % | 12.89 % |
| **Mean** | **63** | **74** | **7.58 %** | **7.32 %** |

### Fold Variance — Pattern and Explanation

The cross-validation results show a bi-modal pattern that requires interpretation:

**Folds 2–4 (sMAPE 1.8 %–4.7 %)** perform well. With 94–192 training observations covering
2002–2014, the model captures the structural dynamics of the FDEFX series reliably.

**Fold 1 (sMAPE 14.7 %)** is expected to underperform: TimeSeriesSplit assigns only ~47
training rows to the first fold, yielding a row-to-feature ratio of ~0.54:1. With 87 features
and fewer than 50 training samples, XGBoost cannot learn stable split thresholds — the result
is methodologically unavoidable, not a model deficiency.

**Fold 5 (sMAPE 13.5 %)** covers approximately 2021–2025 (the final ~48 months of the series).
This period includes the COVID-recovery demand surge, the Ukraine war onset (Feb 2022), and the
subsequent NATO-driven defense spending acceleration — structural breaks not present in the
2002–2020 training window. The deterioration in Fold 5 reflects genuine out-of-distribution
conditions rather than a model failure, but it is a relevant limitation for near-term forecasting.

### Rationale for Target Choice

FDEFX was chosen over ADEFNO to ensure semantic consistency with B1 and B2,
which both forecast invoice-based revenue. ADEFNO is retained as the strongest
leading feature (lags 1–24), allowing the model to learn the order-to-bill
conversion dynamics inherent in defense procurement. The 24-month lag window
is empirically motivated by the heterogeneous lead times across Defense Capital
Goods categories (ammunition: ~3–6 months; aircraft and missiles: ~24 months).

### Note on Metric Magnitude

The mean sMAPE of 7.58 % is lower than the 10.52 % achieved with ADEFNO as target.
**This does not indicate a better model.** FDEFX is a stepped quarterly series
(forward-filled to monthly), which suppresses the high-amplitude month-to-month spikes
characteristic of ADEFNO. A smoother target mechanically yields lower percentage errors.
The directionally correct folds (2–4) confirm that the model captures expenditure trends;
the absolute error magnitude (MAE ≈ 63 USD bn against a series level of ~800–1,000 USD bn)
is acceptable for a market volume proxy used in SAC scenario planning.

### Note on Feature Count

87 features against 287 observations gives a row-to-feature ratio of ~3.3:1. This is
borderline but acceptable for XGBoost, which performs implicit feature selection via
tree-based splits. Fold 1 instability confirms the ratio is tight at small training sizes.
Feature importance (Notebook 05) will reveal which ADEFNO lag horizons dominate — a
concentration in lags 6–18 would empirically confirm the order-to-shipment conversion hypothesis.